# A preference for A5
This notebook solves the project Euler problem 151: <a href="https://projecteuler.net/problem=151">A preference for A5</a>. The notebook performs the following steps:
- Creates an envelope class
- Performs a tree search of all possible combinations of picking a piece of paper
- Calculates the final result and prints

## Detailed explanation
The goal is to calculate the expected number of times that the supervisor finds a single sheet of paper in the envelope, disregarding the first round and last rounds (when there is always just one sheet in the envelope (A1 and A5, respectively)).

The supervisor will always open the envelope 16 times and then it will be empty. Let's keep track of the number of sheets of each size in a variable, `n_per_size`. We count from largest to smallest, i.e. `n_per_size = [n_A1, n_A2, n_A3, n_A4, n_A5]`. From the beginning we have one A1 paper and nothing else, i.e. `n_per_size = [1 0 0 0 0]`. We can then print the evolution of what happens when the supervisor opens the envelope 16 times. Let's say he always picks the biggest sheet in the envelope. Then we have: <br>
`[1 0 0 0 0]`<br>
`[0 1 1 1 1]`<br>
`[0 0 2 2 2]`<br>
`[0 0 1 3 3]`<br>
`[0 0 0 4 4]`<br>
`[0 0 0 3 5]`<br>
`[0 0 0 2 6]`<br>
`[0 0 0 1 7]`<br>
`[0 0 0 0 8]`<br>
`[0 0 0 0 7]`<br>
`[0 0 0 0 6]`<br>
`[0 0 0 0 5]`<br>
`[0 0 0 0 4]`<br>
`[0 0 0 0 3]`<br>
`[0 0 0 0 2]`<br>
`[0 0 0 0 1]`<br>

In this case, the supervisor *never* has an empty envelope. On the other hand, let's see what happens if he always picks the smallest sheet. Now we get:<br>
`[1 0 0 0 0]`<br>
`[0 1 1 1 1]`<br>
`[0 1 1 1 0]`<br>
`[0 1 1 0 1]`<br>
`[0 1 1 0 0]`<br>
`[0 1 0 1 1]`<br>
`[0 1 0 1 0]`<br>
`[0 1 0 0 1]`<br>
`[0 1 0 0 0]` Here the envelope contains only one sheet, an A2<br>
`[0 0 1 1 1]`<br>
`[0 0 1 1 0]`<br>
`[0 0 1 0 1]`<br>
`[0 0 1 0 0]` Here the envelope contains only one sheet, an A3<br>
`[0 0 0 1 1]`<br>
`[0 0 0 1 0]` Here the envelope contains only one sheet, an A4<br>
`[0 0 0 0 1]`<br>

In this case finding an envelope with a single sheet in it happens 3 times. There are of course a multitude of journeys and the question is then, how probably is each possible journey?

Let's create all journeys in a tree. We will then track the number of times we find an empty envelope and how probable each branch is. All we need to do then is to sum the probabilities.

Let's go!

In [1]:
# Set some global variables
RUN_TESTS = False
SCALE_BY_PAPER_SIZE = False

In [2]:
import pandas as pd
import numpy as np

class Envelope:
    expected_value_one_sheet = 0   # The cumulative expected value of the envelope containing one sheet of paper
    
    def __init__(self, n_of_each=np.array([1, 1, 1, 1]), p_branch=1):   # Initialize envelope, n_of_each = [n_A2, n_A3, n_A4, n_A5]
        assert (len(n_of_each) == 4) and all(n_of_each[i] in range(i+2) for i in range(4)), f"n_of_each must be a list of 4 integers no greater than [1, 2, 3, 4], Was {n_of_each}"

        self.n_of_each = n_of_each    # Number of papers of sizes A2, A3, A4 and A5
        self.n_times_opened = 0       # Number of times the envelope is opened
        self.p_branch = 1             # Probability that we are here

    def copy(self):
        out = Envelope()
        out.n_of_each = self.n_of_each
        out.n_times_opened = self.n_times_opened
        out.p_branch = self.p_branch
        return(out)

    def open_and_get_a5(self, ind):   # Open the envelope, get a piece of paper, cut until it is A5
        assert (isinstance(ind, int)) and (ind >= 0) and (ind <= 3), f"ind must be an integer between 0 and 3, was {ind}"
        
        if (self.n_of_each[ind] == 0):    # If we do not have any paper this size, do nothing
            pass
        else:
            # Update the times the envelope has been opened
            self.n_times_opened = self.n_times_opened + 1 

            # Cut the paper(s), if needed
            n_of_each_new = self.n_of_each.copy()
            for i in range(ind+1, 4):
                n_of_each_new[i] = n_of_each_new[i]+1

            # Check if the envelope only contained one sheet of paper
            if (self.n_times_opened != 15) and (sum(self.n_of_each) == 1):
                Envelope.expected_value_one_sheet += self.p_branch

            # Update the object
            n_of_each_new[ind] = n_of_each_new[ind] - 1 
            self.n_of_each = n_of_each_new

    def __repr__(self):
        return f"{self.__class__.__name__}({self.n_of_each}, {self.p_branch})"

    def __str__(self):
        return('Number of times opened: ' + str(self.n_times_opened) + ', n_of_each: ' + str(self.n_of_each) + ', p_branch: ' + str(self.p_branch))

______________
## Some tests

In [3]:
if RUN_TESTS:
    e = Envelope()
    print(e)

In [4]:
if RUN_TESTS:
    e = Envelope()
    print('Instance 1.', e)
    
    e.open_and_get_a5(0)
    print('Instance 1.', e)
    
    e.open_and_get_a5(3)
    print('Instance 1.', e)
    
    e2 = Envelope()
    print('Instance 2.', e2)

In [5]:
if RUN_TESTS:
    # Always taking the smallest sheet. Should result in 3 times the envelope contains only one
    
    e = Envelope()
    inds = [3, 2, 3, 1, 3, 2, 3, 0, 3, 2, 3, 1, 3, 2, 3]
    print(e)
    for i in inds:    
        e.open_and_get_a5(i)
        print(e)
    
    print('\nNumber of times the envelope contained only one sheet:', e.expected_value_one_sheet)

In [6]:
if RUN_TESTS:

    # Always taking the biggest sheet. Should result in 0 times the envelope contains only one
    e2 = Envelope()
    inds = [0, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3]
    print(e2)
    for i in inds:    
        e2.open_and_get_a5(i)
        print(e2)
    
    print('\nNumber of times the envelope contained only one sheet:', e2.expected_value_one_sheet-3)

_______________
## Tree

In [7]:
# Go over the tree
env_list = [Envelope()]    #Initialize
# print(env_list)
done = False
while not done:
    # Take the last envelope (removing it from the list)
    # [print(item) for item in env_list]
    # print('Expected value one sheet, so far:', env_list[-1].expected_value_one_sheet)
    # print(' ')
    e = env_list.pop()

    for i in range(4):
        if e.n_of_each[i] != 0:
            e_new = e.copy()
            # Set probability to pick this branch
            # It is a function of probability so far and, sometimes, size of paper
            if SCALE_BY_PAPER_SIZE:
                p_new = e_new.p_branch * e_new.n_of_each[i] * 2**(3-i)/sum(e_new.n_of_each*np.array([8, 4, 2, 1]))
            else:
                p_new = e_new.p_branch * e_new.n_of_each[i]/sum(e_new.n_of_each)
            e_new.p_branch = p_new

            # Pick a sheet and add pack to list
            e_new.open_and_get_a5(i)
            env_list.append(e_new)

    # Check if the last envelope element is empty = Envelope([0, 0, 0, 0]). 
    # If it is, remove it
    while (len(env_list) > 0) and (all(env_list[-1].n_of_each == [0, 0, 0, 0])):
        env_list.pop()

    # See if we have anything left in the list - if not, then we're done
    if len(env_list) == 0:
        done = True

In [8]:
# Display answer

print(f"The expected number of times the envelope contains 1 sheet is {e.expected_value_one_sheet:.6f}")

The expected number of times the envelope contains 1 sheet is 0.464399
